# 실습 2주차: 층을 쌓아 직선을 벗어나기

> **시나리오 — 오늘 만들 것**
>
>
> 자동차 **392대의 마력(horsepower)** 으로 **연비(mpg)** 를 맞힌다.
>
> 지난주 모형은 **직선**이었다. 그런데 이 관계는 직선이 아니다 —
> 마력이 낮을 때는 연비가 가파르게 떨어지다가, 높아지면 완만해진다.
>
> **직선으로 안 되는 것을 층을 쌓아 푼다.** 오늘 세 모형을 학습시켜 나란히 비교한다.
>
> 1. 퍼셉트론 하나 (= 지난주 모형)
> 2. 두 층인데 **활성화 함수가 없는** 모형
> 3. 두 층 + **ReLU**
>
> - **대응 이론**: [Ch02 퍼셉트론과 다층 퍼셉트론](ch02.qmd)
> - 코드는 완성되어 있다. **직접 해보기** 칸은 스스로 채운 뒤 아래 정답과 맞춰 본다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

---

# 1. 직선으로 안 된다는 것을 눈으로 본다

## 1-1. 데이터 불러오기

In [ ]:
URL = 'https://raw.githubusercontent.com/ralbu85/Lecture_DeepLearning_2022/main/auto.csv'
a = pd.read_csv(URL)
print(a.shape)
a.head()

## 1-2. 마력과 연비의 관계

In [ ]:
plt.figure(figsize=(5.6, 3.8))
plt.scatter(a['horsepower'], a['mpg'], s=14, alpha=0.6)
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**휘어 있다.** 마력 50~100 구간에서는 급하게 떨어지고, 150 이상에서는 거의 평평하다.

## 1-3. 직선을 그어 보면

In [ ]:
from sklearn.linear_model import LinearRegression

X1 = a[['horsepower']].to_numpy(dtype='float32')
y1 = a['mpg'].to_numpy(dtype='float32')

line = LinearRegression().fit(X1, y1)
grid = np.linspace(X1.min(), X1.max(), 200).reshape(-1, 1)

plt.figure(figsize=(5.6, 3.8))
plt.scatter(X1, y1, s=14, alpha=0.5, label='data')
plt.plot(grid, line.predict(grid), 'r-', lw=2, label='straight line')
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

양쪽 끝에서 **체계적으로 빗나간다.** 직선이 표현할 수 있는 모양의 한계다.

> **직접 해보기 ① — 다른 변수도 휘어 있는가**
>
>
> `weight` 와 `mpg` 의 산점도를 그려 보시오. 이것도 직선이 아닌가?

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(5.6, 3.8))
plt.scatter(...)                  # ← 여기를 채우세요
plt.xlabel('weight'); plt.ylabel('mpg')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(5.6, 3.8))
plt.scatter(a['weight'], a['mpg'], s=14, alpha=0.6, color='C1')
plt.xlabel('weight'); plt.ylabel('mpg')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---

# 2. 퍼셉트론 하나

## 2-1. 손으로 계산하면

$$z = w_1 x_1 + w_2 x_2 + \cdots + w_p x_p + b$$

In [ ]:
x = np.array([1.0, 2.0, 3.0])       # 입력 3개
w = np.array([0.5, -1.0, 2.0])      # 가중치 3개
b = 0.1

z = (w * x).sum() + b               # 곱해서 더하고, 편향을 더한다
print('z =', z, '  = 0.5*1 + (-1)*2 + 2*3 + 0.1')

## 2-2. PyTorch로 같은 계산

In [ ]:
layer = nn.Linear(in_features=3, out_features=1)

with torch.no_grad():                       # 우리 숫자를 그대로 넣어 본다
    layer.weight.copy_(torch.tensor([w], dtype=torch.float32))
    layer.bias.copy_(torch.tensor([b], dtype=torch.float32))

out = layer(torch.tensor(x, dtype=torch.float32))
print('nn.Linear :', float(out))
print('손계산    :', z)

> **`nn.Linear` 의 가중치 모양은 `(출력, 입력)` 이다**
>
>
> ```python
> nn.Linear(3, 1).weight.shape   # → (1, 3)
> ```
>
> 수식은 $x W^\top + b$ 로 계산된다. 순서를 헷갈리면 shape 에러가 난다.


## 2-3. 여러 대를 한 번에

In [ ]:
X = torch.tensor(np.random.randn(5, 3), dtype=torch.float32)   # 자동차 5대
print('입력 :', tuple(X.shape))
print('출력 :', tuple(layer(X).shape), '  ← 대당 하나씩')
print(layer(X).detach().numpy().round(3))

> **직접 해보기 ② — 입력 4개짜리 퍼셉트론**
>
>
> 입력 4개, 출력 1개인 퍼셉트론을 만들고, 자동차 10대짜리 입력을 통과시켜 출력 shape을 확인하시오.

In [ ]:
# ✏️ 직접 채워 보세요
p = None                    # ← nn.Linear(...)
X10 = torch.randn(10, 4)
out10 = None                # ← p(X10)

assert out10 is not None and tuple(out10.shape) == (10, 1), '모양을 확인하세요'
print('통과  출력 shape', tuple(out10.shape),
      ' 파라미터', sum(q.numel() for q in p.parameters()))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
p = nn.Linear(4, 1)
X10 = torch.randn(10, 4)
out10 = p(X10)
print('출력 shape', tuple(out10.shape),
      ' 파라미터', sum(q.numel() for q in p.parameters()), '= 4*1 + 1')

---

# 3. 노드를 여러 개로 — 층

퍼셉트론 하나는 숫자 하나를 낸다. **여러 개를 나란히 놓은 것**이 층이다.

In [ ]:
hidden = nn.Linear(1, 8)          # 입력 1개 → 노드 8개

print('weight :', tuple(hidden.weight.shape), ' = (출력, 입력)')
print('bias   :', tuple(hidden.bias.shape))
print('파라미터:', sum(p.numel() for p in hidden.parameters()), ' = 1*8 + 8')

x1 = torch.tensor([[100.0]])      # 마력 100인 자동차 한 대
print('\n입력:', tuple(x1.shape), '→ 출력:', tuple(hidden(x1).shape))
print(hidden(x1).detach().numpy().round(3))

노드 8개가 **같은 입력을 서로 다른 가중치로** 본다. 8개의 서로 다른 관점이 생긴 것이다.

> **직접 해보기 ③ — 파라미터 수를 먼저 예측하기**
>
>
> `nn.Linear(5, 12)` 의 파라미터 수는 몇 개인가? **먼저 계산한 뒤** 확인하시오.

In [ ]:
# ✏️ 직접 채워 보세요
my_answer = None            # ← 예상한 숫자를 적으세요

real = sum(p.numel() for p in nn.Linear(5, 12).parameters())
assert my_answer == real, f'다릅니다. 실제는 {real} — 공식은 (입력 x 출력) + 출력'
print('정답', real)

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
my_answer = 5 * 12 + 12
real = sum(p.numel() for p in nn.Linear(5, 12).parameters())
print('예상', my_answer, ' 실제', real, ' 공식 (입력 x 출력) + 출력')

---

# 4. 활성화 함수 — 층을 쌓는 의미를 만드는 것

## 4-1. 원소마다 적용된다

In [ ]:
z = torch.tensor([[-2.0, -0.5, 0.0, 1.5, 3.0]])

print('입력    :', z.numpy()[0])
print('ReLU    :', nn.ReLU()(z).numpy()[0])
print('Sigmoid :', nn.Sigmoid()(z).numpy()[0].round(4))
print('Tanh    :', nn.Tanh()(z).numpy()[0].round(4))
print('\n파라미터 수:', sum(p.numel() for p in nn.ReLU().parameters()), ' ← 배우는 것이 없다')

In [ ]:
t = torch.linspace(-4, 4, 200)
plt.figure(figsize=(6, 3.2))
for f, name in [(nn.ReLU(), 'ReLU'), (nn.Sigmoid(), 'Sigmoid'), (nn.Tanh(), 'Tanh')]:
    plt.plot(t, f(t), label=name)
plt.axhline(0, lw=0.6); plt.axvline(0, lw=0.6)
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4-2. 활성화가 없으면 층을 쌓아도 층 하나다

두 층을 이어 붙이되 **활성화를 넣지 않으면** 어떻게 되는지 직접 확인한다.

In [ ]:
torch.manual_seed(0)
L1 = nn.Linear(3, 8, bias=False)
L2 = nn.Linear(8, 2, bias=False)

X = torch.randn(4, 3)
two_layers = L2(L1(X))                       # 두 층을 통과

W_combined = L2.weight @ L1.weight           # (2,8) @ (8,3) → (2,3)
one_layer = X @ W_combined.T                 # 층 하나로 같은 계산

print('두 층 통과:\n', two_layers.detach().numpy().round(4))
print('\n층 하나로 :\n', one_layer.detach().numpy().round(4))
print('\n같은가:', torch.allclose(two_layers, one_layer, atol=1e-5))
print('합쳐진 가중치 shape:', tuple(W_combined.shape), ' ← 결국 (3 → 2) 한 층')

> **이것이 활성화 함수가 필요한 이유**
>
>
> $$W_2(W_1 x) = (W_2 W_1) x$$
>
> 행렬 두 개를 곱하면 **행렬 하나**다. 활성화 없이 층을 100개 쌓아도 결국 **직선 하나**다.
> 중간에 ReLU처럼 **직선이 아닌 함수**를 끼워야 비로소 층이 층 노릇을 한다.


> **직접 해보기 ④ — ReLU를 넣으면 깨지는가**
>
>
> 위 두 층 사이에 `nn.ReLU()` 를 넣고, 여전히 `X @ W_combined.T` 와 같은지 확인하시오.

In [ ]:
# ✏️ 직접 채워 보세요
with_relu = None            # ← L2(nn.ReLU()(L1(X)))

print('ReLU 넣은 결과:\n', with_relu.detach().numpy().round(4))
print('층 하나와 같은가:', torch.allclose(with_relu, one_layer, atol=1e-5))

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
with_relu = L2(nn.ReLU()(L1(X)))
print('ReLU 넣은 결과:\n', with_relu.detach().numpy().round(4))
print('층 하나와 같은가:', torch.allclose(with_relu, one_layer, atol=1e-5), ' ← 더 이상 같지 않다')

---

# 5. 층 쌓기 — `nn.Sequential`

In [ ]:
mlp = nn.Sequential(
    nn.Linear(1, 16),      # 마력 1개 → 은닉 노드 16개
    nn.ReLU(),
    nn.Linear(16, 1),      # 은닉 16개 → 연비 1개
)
print(mlp)

In [ ]:
h = torch.zeros(5, 1)
for i, layer in enumerate(mlp):
    h = layer(h)
    print(f'{i} {layer.__class__.__name__:8s} → {tuple(h.shape)}  '
          f'파라미터 {sum(p.numel() for p in layer.parameters())}')
print('\n전체 파라미터:', sum(p.numel() for p in mlp.parameters()), ' = (1*16+16) + (16*1+1)')

---

# 6. 완성 — 세 모형을 학습시켜 비교한다

## 6-1. 준비

In [ ]:
from sklearn.model_selection import train_test_split

X = a[['horsepower']].to_numpy(dtype='float32')
y = a['mpg'].to_numpy(dtype='float32')

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

mu, sd = X_tr.mean(0), X_tr.std(0)          # 통계는 훈련 데이터에서만
ym, ys = y_tr.mean(), y_tr.std()

T = lambda A, b: (torch.tensor((A - mu) / sd),
                  torch.tensor((b - ym) / ys).unsqueeze(1))
Xt_tr, yt_tr = T(X_tr, y_tr)
Xt_te, yt_te = T(X_te, y_te)

print('훈련', X_tr.shape[0], '대   시험', X_te.shape[0], '대')
print('입력 텐서', tuple(Xt_tr.shape), '  정답 텐서', tuple(yt_tr.shape))

## 6-2. 학습 함수

In [ ]:
def fit(model, X, y, X_val, y_val, epochs=1000, lr=0.05):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    hist = {'train': [], 'val': []}
    for ep in range(epochs):
        optimizer.zero_grad()          # ① 기울기 초기화
        loss = criterion(model(X), y)  # ② 예측하고 손실 계산
        loss.backward()                # ③ 기울기 구하기
        optimizer.step()               # ④ 가중치 갱신
        hist['train'].append(loss.item())
        with torch.no_grad():
            hist['val'].append(criterion(model(X_val), y_val).item())
    return hist


def test_rmse(model):
    with torch.no_grad():
        pred = model(Xt_te).squeeze(1).numpy() * ys + ym
    return float(np.sqrt(((pred - y_te) ** 2).mean()))

> 이 네 줄(①~④)이 학습의 전부다. **3주차에 직접 만든다.**

## 6-3. 세 모형을 학습시킨다

In [ ]:
models = {
    '1층 (선형)':      lambda: nn.Linear(1, 1),
    '2층 활성화 없음': lambda: nn.Sequential(nn.Linear(1, 16), nn.Linear(16, 1)),
    '2층 + ReLU':      lambda: nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1)),
}

trained, hists, rows = {}, {}, []
for name, make in models.items():
    torch.manual_seed(42)
    m = make()
    hists[name] = fit(m, Xt_tr, yt_tr, Xt_te, yt_te)
    trained[name] = m
    rows.append({'모형': name,
                 '파라미터': sum(p.numel() for p in m.parameters()),
                 '테스트 RMSE (mpg)': round(test_rmse(m), 3)})

print(pd.DataFrame(rows).to_string(index=False))

> **앞의 두 줄을 보라 — 숫자가 **똑같다****
>
>
> 파라미터는 2개와 49개로 24배 차이인데 성능은 **소수점까지 같다.**
> 4-2절에서 증명한 그대로 — 활성화 없는 2층은 **선형 모형 그 자체**다.


## 6-4. 러닝커브

In [ ]:
plt.figure(figsize=(6.2, 3.6))
for name in models:
    plt.plot(hists[name]['val'], label=name)
plt.xlabel('epoch'); plt.ylabel('validation MSE (standardized)')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6-5. 무엇을 그렸는지 눈으로 확인한다

In [ ]:
grid = np.linspace(X.min(), X.max(), 300).reshape(-1, 1).astype('float32')
gt = torch.tensor((grid - mu) / sd)

plt.figure(figsize=(6.4, 4.2))
plt.scatter(X, y, s=14, alpha=0.35, color='gray', label='data')
for name, m in trained.items():
    with torch.no_grad():
        curve = m(gt).squeeze(1).numpy() * ys + ym
    plt.plot(grid, curve, lw=2, label=name)
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

앞의 두 모형은 **직선 하나로 겹쳐 있고**, ReLU를 넣은 모형만 **꺾인 선**으로 데이터를 따라간다.

> ReLU는 꺾인 직선이다. 그것을 16개 겹치면 **꺾은선으로 곡선을 흉내낼 수 있다.**
> 은닉 노드를 늘릴수록 더 촘촘한 꺾은선이 된다.


> **직접 해보기 ⑤ — 은닉 노드 수를 바꿔 보기**
>
>
> 은닉 노드를 `2, 4, 16, 64` 로 바꿔 학습시키고, 예측 곡선을 한 그림에 겹쳐 보시오.
> 노드가 적으면 곡선이 어떻게 되는가?

In [ ]:
# ✏️ 직접 채워 보세요
plt.figure(figsize=(6.4, 4.2))
plt.scatter(X, y, s=14, alpha=0.3, color='gray')
for h in [2, 4, 16, 64]:
    torch.manual_seed(42)
    m = ...                       # ← nn.Sequential(...)로 은닉 h개짜리 모형
    fit(m, Xt_tr, yt_tr, Xt_te, yt_te)
    with torch.no_grad():
        plt.plot(grid, m(gt).squeeze(1).numpy() * ys + ym, lw=2, label=f'hidden {h}')
plt.xlabel('horsepower'); plt.ylabel('mpg'); plt.legend(fontsize=8); plt.show()

In [ ]:
# ↓ 정답 코드 (먼저 스스로 해 본 뒤에 실행하세요)
plt.figure(figsize=(6.4, 4.2))
plt.scatter(X, y, s=14, alpha=0.3, color='gray')
for h in [2, 4, 16, 64]:
    torch.manual_seed(42)
    m = nn.Sequential(nn.Linear(1, h), nn.ReLU(), nn.Linear(h, 1))
    fit(m, Xt_tr, yt_tr, Xt_te, yt_te)
    with torch.no_grad():
        plt.plot(grid, m(gt).squeeze(1).numpy() * ys + ym, lw=2,
                 label=f'hidden {h}  (RMSE {test_rmse(m):.2f})')
plt.xlabel('horsepower'); plt.ylabel('mpg')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

노드가 2개면 꺾이는 곳이 하나뿐이라 곡선을 따라가지 못한다.
**은닉 노드 수는 모형이 표현할 수 있는 모양의 복잡도**를 정한다 — 하이퍼파라미터다.

---

# 7. 정리

> **이번 주 체크포인트**
>
>
> | 하고 싶은 일 | 코드 | 결과 |
> |------|------|------|
> | 퍼셉트론 하나 | `nn.Linear(p, 1)` | 파라미터 $p+1$ |
> | 층 (노드 여러 개) | `nn.Linear(p, h)` | 파라미터 $p\cdot h + h$ |
> | 가중치 모양 | `layer.weight.shape` | `(출력, 입력)` |
> | 활성화 | `nn.ReLU()`, `nn.Sigmoid()`, `nn.Tanh()` | 파라미터 0 |
> | 층 쌓기 | `nn.Sequential(...)` | |
> | 층별 shape 확인 | `for layer in model:` 하나씩 통과 | |
> | 학습 4줄 | `zero_grad → loss → backward → step` | |


**오늘의 결론**

$$W_2(W_1x) = (W_2W_1)x \quad\Longrightarrow\quad \text{활성화가 없으면 층을 쌓아도 직선}$$

## 스스로 확인해 보기

아래 결과를 먼저 예상한 뒤 실행해서 대조한다.

In [ ]:
net = nn.Sequential(
    nn.Linear(4, 10), nn.ReLU(),
    nn.Linear(10, 6), nn.ReLU(),
    nn.Linear(6, 3),
)
x = torch.randn(7, 4)

h = x
for i, layer in enumerate(net):
    h = layer(h)
    print(f'{i} {layer.__class__.__name__:8s} → {tuple(h.shape)}')

print('\n전체 파라미터:', sum(p.numel() for p in net.parameters()))
print('손으로:', (4*10+10), '+', (10*6+6), '+', (6*3+3), '=', (4*10+10)+(10*6+6)+(6*3+3))

---

## 다음 실습

[실습 3주차: 학습 루프를 직접 만든다](lab03.qmd) —
오늘 그냥 쓴 `fit()` 함수를 **한 줄씩 직접 만든다.**
손실 함수가 무엇이고, `backward()` 가 무엇을 계산하는지 열어 본다.